# Assigment 4

In [159]:
import math
import numpy as np
import scipy.sparse as sp

import igl
import meshplot as mp

from math import sqrt

# Notes


In [160]:
v, f = igl.read_triangle_mesh("data/irr4-cyl2.off")
tt, _ = igl.triangle_triangle_adjacency(f)

c = np.loadtxt("data/irr4-cyl2.constraints")
cf = c[:, 0].astype(np.int64)
c = c[:, 1:]

In [161]:
def align_field(V, F, TT, soft_id, soft_value, llambda):
    assert(soft_id[0] > 0)
    assert(soft_id.shape[0] == soft_value.shape[0])


    # Edges
    e1 = V[F[:, 1], :] - V[F[:, 0], :]
    e2 = V[F[:, 2], :] - V[F[:, 0], :]

    # Compute the local reference systems for each face, T1, T2
    T1 = e1 / np.linalg.norm(e1, axis=1)[:,None]

    T2 =  np.cross(T1, np.cross(T1, e2))
    T2 /= np.linalg.norm(T2, axis=1)[:,None]

    # Arrays for the entries of the matrix
    data = []
    ii = []
    jj = []

    index = 0
    for f in range(F.shape[0]):
        for ei in range(3): # Loop over the edges

            # Look up the opposite face
            g = TT[f, ei]

            # If it is a boundary edge, it does not contribute to the energy
            # or avoid to count every edge twice
            if g == -1 or f > g:
                continue

            # Compute the complex representation of the common edge
            e  = V[F[f, (ei+1)%3], :] - V[F[f, ei], :]

            vef = np.array([np.dot(e, T1[f, :]), np.dot(e, T2[f, :])])
            vef /= np.linalg.norm(vef)
            ef = (vef[0] + vef[1]*1j).conjugate()

            veg = np.array([np.dot(e, T1[g, :]), np.dot(e, T2[g, :])])
            veg /= np.linalg.norm(veg)
            eg = (veg[0] + veg[1]*1j).conjugate()


            # Add the term conj(f)^n*ui - conj(g)^n*uj to the energy matrix
            data.append(ef);  ii.append(index); jj.append(f)
            data.append(-eg); ii.append(index); jj.append(g)

            index += 1


    sqrtl = sqrt(llambda)

    # Convert the constraints into the complex polynomial coefficients and add them as soft constraints

    # Rhs of the system
    b = np.zeros(index + soft_id.shape[0], dtype=complex)

    for ci in range(soft_id.shape[0]):
        f = soft_id[ci]
        v = soft_value[ci, :]

        # Project on the local frame
        c = np.dot(v, T1[f, :]) + np.dot(v, T2[f, :])*1j

        data.append(sqrtl); ii.append(index); jj.append(f)
        b[index] = c * sqrtl

        index += 1

    assert(b.shape[0] == index)


    # Solve the linear system
    A = sp.coo_matrix((data, (ii, jj)), shape=(index, F.shape[0])).asformat("csr")
    u = sp.linalg.spsolve(A.conjugate().T @ A, A.conjugate().T @ b)

    R = T1 * u.real[:,None] + T2 * u.imag[:,None]

    return R

In [162]:
def plot_mesh_field(V, F, R, constrain_faces):
    # Highlight in red the constrained faces
    col = np.ones_like(f)
    col[constrain_faces, 1:] = 0
    
    # Scaling of the representative vectors
    avg = igl.avg_edge_length(V, F)/2

    #Plot from face barycenters
    B = igl.barycenter(V, F)

    p = mp.plot(V, F, c=col)
    p.add_lines(B, B + R * avg)
    
    return p

In [163]:
R = align_field(v, f, tt, cf, c, 1e6)
plot_mesh_field(v, f, R, cf)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

# 2. Reconstructing a scalar field from a vector field

In [164]:


def scalar_field(V, F, u):
    
    G = igl.grad(V, F)
    # area matrix
    A = sp.diags(np.tile((igl.doublearea(V, F) / 2), 3))
    
    # system
    K = G.T @ A @ G
    b = 2 * G.T @ A @ u 
    
    K = K.T + K
    # make first diagonal 1
    K[0, 0] = 1
    
    # solve
    scalar = sp.linalg.spsolve(K, b)
    
    gradient = G @ scalar
    gradient = gradient.reshape(F.shape, order = "F")
    return scalar, gradient

In [165]:
scalar, gradient = scalar_field(v, f, R.T.flatten())
poisson_error = np.linalg.norm(R - gradient, ord=2, axis=1)

# plotting 
avg = igl.avg_edge_length(v, f)/2
B = igl.barycenter(v, f)

p = mp.plot(v, f, c=scalar)
p.add_lines(B, B + gradient * avg)

p = mp.plot(v, f, c=poisson_error)
print("Poisson Errorr: ", sum(poisson_error))


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Poisson Errorr:  170.4749730072472


# 3. Harmonic and LSCM Parameterizations - Optional

In [166]:
v, f = igl.read_triangle_mesh("data/camel_head.off")

In [167]:
def harmonic(V, F):
    G = igl.grad(V, F)
    # Harmonic mapping
    bnd = igl.boundary_loop(F) # boundary loop
    bnd_uv = igl.map_vertices_to_circle(V, bnd)
    uv = igl.harmonic_weights(V, F, bnd, bnd_uv, 1)
    
    grad = np.reshape(G @ uv[:, 1], F.shape, order='F')
    return uv, grad


In [168]:
def LSCM(V, F):
    bnd = igl.boundary_loop(F) # boundary loop
    b = np.array([bnd[0], bnd[len(bnd) // 2]])
    bc = np.array([[0.0, 0.0], [1.0, 0.0]])
    _, uv = igl.lscm(V, F, b, bc)
    return uv

In [169]:
def plot_harmonic_and_uv(v, f, uv, g):
    print("Harmonic param and uv: ")
    p = mp.subplot(v, f, s=[1, 2, 0], shading={"wireframe": True})
    mp.subplot(uv, f, shading={"wireframe": True}, s=[1, 2, 1], data=p)

    print("Harmonic vector gradient field")
    avg = igl.avg_edge_length(v, f) / 2
    B = igl.barycenter(v, f)
    p2 = mp.plot(v, f, shading={"wireframe": True})
    p2.add_lines(B, B + g * avg, shading={"line_color": "blue"})
    
def plot_lscm_and_uv(v, f, uv):
    print("LSCM")
    p = mp.subplot(v, f,  s=[1, 2, 0], shading={"wireframe": True})
    mp.subplot(uv, f, shading={"wireframe": True}, s=[1, 2, 1], data=p)

In [170]:
# Harmonic 
uv_harmonic, g_harmonic = harmonic(v, f)
plot_harmonic_and_uv(v, f, uv_harmonic, g_harmonic)

# LSCM 
uv_LSCM = LSCM(v, f)
plot_lscm_and_uv(v, f, uv_LSCM)

Harmonic param and uv: 


Harmonic vector gradient field


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(1.9967555…

LSCM


# 4. Editing a parameterization with vector fields - Optional 

In [171]:
v, f = igl.read_triangle_mesh("data/irr4-cyl2.off")
tt, _ = igl.triangle_triangle_adjacency(f)

c = np.loadtxt("data/irr4-cyl2.constraints")
cf = c[:, 0].astype(np.int64)
c = c[:, 1:]